# W.B. Yeats — Workshop Notebook
**CompLit 126x — Love in Context**

This notebook implements the prompt chain designed by the W.B. Yeats workshop group. Their chain used a **linguistic trait extraction + iterative polish** architecture — feeding in actual Yeats poems, analyzing them in two separate passes (linguistic traits, then formal/image traits), and bouncing between models for critique and synthesis.

```
5 Yeats poems          analyze common          generate
(selected randomly) ──▶ linguistic traits ──▶  Yeats-like sonnet
        │                                           │
        │                                           ▼
        │                                      judgements +
        │                                      polishing advice
        │                                      (fresh context)
        │                                           │
        └──▶ learn formal     ◀─────────────────────┘
             and images traits
                    │
                    ▼
             generate sonnet,
             synthesizing ALL     ──▶  final polish  ──▶  final
             previous analysis         (fresh context)     ver.!
```

**What makes this chain interesting:** The group split their analysis into two distinct passes — **linguistic traits** first (diction, syntax, sound patterns), then **formal and image traits** (structure, imagery, symbolism) — with a critique step in between. This layered approach meant the model built understanding incrementally rather than trying to analyze everything at once. They also intentionally chose poems "randomly" rather than picking the most famous ones, to avoid the model falling back on its strongest associations.

The group originally used Gemini for generation/analysis and ChatGPT for critique. This notebook uses GPT-4o throughout, simulating the cross-model effect with separate API calls.

---
**Run the cells in order.** Each step builds on the previous one.

## Setup
Run the two cells below once at the start of your session.

In [ ]:
# Install the OpenAI SDK (run once per session)
%pip install openai --quiet
print("✓ Installed")

In [ ]:
from openai import OpenAI
import json
import os

# ── API Key ──────────────────────────────────────────────────────────────────
# In Google Colab:
#   1. Click the 🔑 (Secrets) icon in the left sidebar
#   2. Add a secret named  OPENAI_API_KEY  with your key
#   3. Toggle "Notebook access" to ON, then run this cell
#
# Locally: set the OPENAI_API_KEY environment variable

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    print("✓ Using Colab Secrets")
except (ImportError, Exception):
    api_key = os.environ.get('OPENAI_API_KEY')
    print("✓ Using environment variable")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o"

# Helper: call the model and return the text
def ask(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.responses.create(model=MODEL, input=messages)
    return response.output_text

print(f"✓ Client ready | Model: {MODEL}")

---
## Step 1: Feed In Yeats Poems

The group selected five Yeats poems "randomly" — deliberately avoiding the most anthologized pieces to prevent the model from falling back on its strongest (and most generic) associations. They chose a mix of longer and shorter poems to give the model range.

**Paste your Yeats poems below.** Pick 5 that span different periods or moods. The group's instinct to choose somewhat randomly is good — it forces the model to find patterns rather than confirm what it already "knows."

In [ ]:
# ── Step 1: Feed in Yeats poems ─────────────────────────────────────────────
# Paste actual Yeats poems here. The group used 5, selected randomly.
# Replace these with your own selections.

YEATS_POEMS = [
    {
        "title": "When You Are Old",
        "text": """When you are old and grey and full of sleep,
And nodding by the fire, take down this book,
And slowly read, and dream of the soft look
Your eyes had once, and of their shadows deep;

How many loved your moments of glad grace,
And loved your beauty with love false or true,
But one man loved the pilgrim soul in you,
And loved the sorrows of your changing face;

And bending down beside the glowing bars,
Murmur, a little sadly, how Love fled
And paced upon the mountains overhead
And hid his face amid a crowd of stars."""
    },
    {
        "title": "The Wild Swans at Coole",
        "text": """The trees are in their autumn beauty,
The woodland paths are dry,
Under the October twilight the water
Mirrors a still sky;
Upon the brimming water among the stones
Are nine-and-fifty swans.

The nineteenth autumn has come upon me
Since I first made my count;
I saw, before I had well finished,
All suddenly mount
And scatter wheeling in great broken rings
Upon their clamorous wings."""
    },
    {
        "title": "No Second Troy",
        "text": """Why should I blame her that she filled my days
With misery, or that she would of late
Have taught to ignorant men most violent ways,
Or hurled the little streets upon the great,
Had they but courage equal to desire?
What could have made her peaceful with a mind
That nobleness made simple as a fire,
With beauty like a tightened bow, a kind
That is not natural in an age like this,
Being high and solitary and most stern?
Why, what could she have done, being what she is?
Was there another Troy for her to burn?"""
    },
    {
        "title": "Adam's Curse",
        "text": """We sat together at one summer's end,
That beautiful mild woman, your close friend,
And you and I, and talked of poetry.
I said, 'A line will take us hours maybe;
Yet if it does not seem a moment's thought,
Our stitching and unstitching has been naught.
Better go down upon your marrow-bones
And scrub a kitchen pavement, or break stones
Like an old pauper, in all kinds of weather;
For to articulate sweet sounds together
Is to work harder than all these, and yet
Be thought an idler by the noisy set
Of bankers, schoolmasters, and clergymen
The martyrs call the world.'"""
    },
    {
        "title": "The Second Coming",
        "text": """Turning and turning in the widening gyre
The falcon cannot hear the falconer;
Things fall apart; the centre cannot hold;
Mere anarchy is loosed upon the world,
The blood-dimmed tide is loosed, and everywhere
The ceremony of innocence is drowned;
The best lack all conviction, while the worst
Are full of passionate intensity."""
    }
]

poems_text = "\n\n---\n\n".join(
    f"\"{p['title']}\":\n{p['text']}" for p in YEATS_POEMS
)

print(f"✓ Loaded {len(YEATS_POEMS)} Yeats poems")
for p in YEATS_POEMS:
    print(f"  · \"{p['title']}\" ({len(p['text'].split())} words)")

---
## Step 2: Analyze Common Linguistic Traits

The first analytical pass: **linguistic traits**. This focuses on Yeats's language at the level of diction, syntax, and sound — *how* he writes, not *what* he writes about. The group kept this separate from formal/image analysis to force more detailed attention to each layer.

In [ ]:
# ── Step 2: Analyze linguistic traits ────────────────────────────────────────

linguistic = ask(
    f"""Here are five poems by W.B. Yeats:

{poems_text}

Analyze the common LINGUISTIC TRAITS across these poems. Focus on:

- **Diction**: What kind of words does Yeats favor? (Anglo-Saxon vs.
  Latinate, concrete vs. abstract, archaic vs. modern)
- **Syntax**: How does he build sentences? (inversions, subordinate
  clauses, rhetorical questions, imperative mood)
- **Sound patterns**: Alliteration, assonance, internal rhyme,
  consonance — what sounds recur?
- **Register**: How formal is the language? Where does he shift
  between the elevated and the plain?
- **Characteristic phrases**: Patterns like "the X of Y," compound
  adjectives, particular verb choices

Be specific. Quote examples from the poems above.
This is about LANGUAGE, not theme or imagery — we'll get to those next."""
)

print("LINGUISTIC TRAITS")
print("═" * 60)
print(linguistic)

---
## Step 3: Generate Yeats-Like Sonnet (First Pass)

Now generate a sonnet using *only* the linguistic traits. This is a partial generation — we're testing whether linguistic analysis alone can produce something Yeats-like, before adding the formal and image layers.

In [ ]:
# ── Step 3: Generate from linguistic traits ──────────────────────────────────

first_sonnet = ask(
    f"""Here is an analysis of W.B. Yeats's linguistic traits:

{linguistic}

Using these linguistic traits as your guide, write a sonnet in the
style of Yeats. Focus on matching his LANGUAGE — his diction, syntax,
sound patterns, and register.

The poem should sound like Yeats wrote it, at the level of the
individual word and sentence. Don't worry yet about matching his
imagery or formal structure perfectly — focus on the language.

Write only the poem."""
)

print("FIRST SONNET (from linguistic traits)")
print("═" * 60)
print(first_sonnet)

---
## Step 4: Judgements + Polishing Advice

The group switched to ChatGPT for this step — a fresh model to judge the sonnet and give polishing advice. We simulate this with a clean API call using a critic system prompt. The fresh context sees only the poem and the actual Yeats, not the analysis that produced it.

In [ ]:
# ── Step 4: Judgements + polishing advice ────────────────────────────────────
# Fresh context — simulates switching to ChatGPT.

polishing = ask(
    f"""Here is a sonnet that attempts to imitate W.B. Yeats's style:

{first_sonnet}

And here are actual Yeats poems for comparison:

{poems_text}

Give your judgement of this imitation and specific polishing advice:

1. What does it get right about Yeats's voice?
2. Where does it fall short — which lines sound un-Yeatsian?
3. Is the language natural or does it feel forced / too intentional
   in its use of Yeats's traits?
4. Specific polishing advice: what 3–5 concrete changes would make
   this sound more like Yeats and more like a real poem?
5. The group noted that AI poems tend to use imagery "too intentionally."
   Does this poem have that problem? Where?

Be direct and quote specific lines.""",
    system="You are a poetry critic and Yeats scholar. Be honest and specific. Focus on what would make the poem feel more natural and human."
)

print("JUDGEMENTS + POLISHING ADVICE")
print("═" * 60)
print(polishing)

---
## Step 5: Learn Formal and Image Traits (Second Pass)

Now the second analytical pass: **formal structure and imagery**. The group fed the Yeats poems back to the model and asked it to analyze a different layer — not language this time, but how Yeats constructs a poem visually and architecturally. This separation forces the model to build a deeper, layered understanding.

In [ ]:
# ── Step 5: Analyze formal and image traits ──────────────────────────────────

formal_images = ask(
    f"""Here are five poems by W.B. Yeats:

{poems_text}

Analyze the FORMAL STRUCTURE and IMAGERY across these poems.
This is a separate analysis from linguistic traits — focus on:

**Formal structure:**
- Stanza patterns and line lengths
- Rhyme schemes (exact, slant, or absent)
- How poems open and close — what kinds of beginnings and endings
- The "turn" — where the poem shifts in argument or feeling
- Relationship between form and content (does the form enact
  the meaning?)

**Imagery:**
- Yeats's characteristic images (swans, towers, gyres, fire, moon,
  stone, birds, etc.)
- How he uses images — as symbols? as arguments? as decoration?
- The ratio of concrete to abstract imagery
- How images develop across a poem (do they accumulate, transform,
  or resolve?)
- The "Yeats image" vs. a generic poetic image — what's the difference?

Be specific. Quote from the poems."""
)

print("FORMAL + IMAGE TRAITS")
print("═" * 60)
print(formal_images)

---
## Step 6: Generate Sonnet Synthesizing All Analysis

Now the big synthesis: combine the linguistic traits, the formal/image traits, and the polishing advice into a single generation. This is where the two-pass approach pays off — the model has a layered understanding rather than a flat one.

The group specifically asked the model to keep the poem "natural and human-like" and not to "intentionally use a large amount of images" — they'd noticed the model over-applied Yeats's imagery.

In [ ]:
# ── Step 6: Synthesize all analysis ──────────────────────────────────────────

synthesis = ask(
    f"""Write a new sonnet in the style of W.B. Yeats, synthesizing
ALL of the following analysis:

LINGUISTIC TRAITS:
{linguistic}

FORMAL AND IMAGE TRAITS:
{formal_images}

POLISHING ADVICE (from critique of a previous attempt):
{polishing}

Important guidelines:
- Keep the language NATURAL and human-like — do not intentionally
  cram in Yeatsian images or archaic diction. Let them emerge.
- Do not use imagery "too intentionally" — a Yeats poem doesn't
  announce its symbols; they grow from the situation.
- The poem should feel like it has a real occasion — someone
  thinking about love, loss, or beauty in a specific moment.
- Match Yeats's balance of the elevated and the plain, the mythic
  and the personal.

Write only the poem."""
)

print("SYNTHESIZED SONNET")
print("═" * 60)
print(synthesis)

---
## Step 7: Final Polish

One last pass — back to a fresh context for final polishing. The group sent the synthesized sonnet to ChatGPT for a final round of advice, then produced the definitive version.

In [ ]:
# ── Step 7a: Final polishing advice ──────────────────────────────────────────
# Fresh context again — only sees the poem and the actual Yeats.

final_advice = ask(
    f"""Here is a sonnet written in the style of W.B. Yeats:

{synthesis}

And here are actual Yeats poems:

{poems_text}

This is a final polishing pass. The poem has already been through
several rounds of analysis and revision. Give concise, specific
advice for the last 10% of improvement:

- Which lines are strongest? Keep them.
- Which 2–3 lines are weakest? Suggest specific replacements.
- Does the poem sound human or does it sound generated?
  Where specifically?
- Any word-level changes (swap one word for another) that would help?

Be brief and actionable.""",
    system="You are a poetry editor. Be concise and give specific line-level advice."
)

print("FINAL POLISHING ADVICE")
print("═" * 60)
print(final_advice)

In [ ]:
# ── Step 7b: Final version ───────────────────────────────────────────────────

final_poem = ask(
    f"""Here is a sonnet in the style of W.B. Yeats:

{synthesis}

Here is final polishing advice:

{final_advice}

Apply the advice. Keep the strongest lines unchanged. Fix the
weakest ones. Make the word-level swaps suggested. The result should
sound natural and human — like a real poem, not a generated one.

Write only the final poem."""
)

print("FINAL VERSION ✦")
print("═" * 60)
print(final_poem)

---
## Compare All Versions

The progression shows the layered approach: linguistic analysis produced a first attempt, then formal/image analysis deepened the model's understanding, and the synthesis brought everything together.

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────

print("PROGRESSION")
print("\n" + "═" * 60)
print("1. FIRST SONNET (linguistic traits only)")
print("═" * 60)
print(first_sonnet)

print("\n" + "═" * 60)
print("2. SYNTHESIZED (linguistic + formal/image + polishing advice)")
print("═" * 60)
print(synthesis)

print("\n" + "═" * 60)
print("3. FINAL VERSION (after last polish)")
print("═" * 60)
print(final_poem)

print("\n" + "═" * 60)
print("\nFor your essay, consider:")
print("  → What did separating linguistic from formal/image analysis add?")
print("  → Did the first sonnet (language only) feel like Yeats?")
print("    What was missing?")
print("  → Did the synthesis feel 'too intentional' in its imagery?")
print("  → The group noted AI poems lack 'conscious logic' between")
print("    images. Do you see that here?")
print("  → What's the difference between sounding like Yeats and")
print("    thinking like Yeats?")

---
## Going Further

This chain is a starting point. Here are ways to extend it for your assignment:

**Try different poem selections.** The group chose randomly. What if you chose only early Yeats? Only late Yeats? Only love poems? The selection shapes the analysis.

**Add a third analytical pass.** Linguistic traits, formal/image traits — what about **thematic traits**? Or **rhetorical moves**? Each new pass deepens the model's understanding.

**Test the "intentionality" problem.** Generate 5 sonnets from the synthesized analysis. Look at the imagery in each. Is there a pattern of over-application? Can you prompt the model to be *less* intentional?

**Compare early vs. late Yeats.** Run the full chain twice — once with early poems ("The Lake Isle of Innisfree," "When You Are Old") and once with late poems ("The Circus Animals' Desertion," "Under Ben Bulben"). Do you get two different Yeatses?

**Generate love song lyrics.** Yeats's elevated register might resist the song form — or it might produce something surprisingly powerful. Try it.

**Submitting your work:**
- **Lyrics**: Submit an album's worth of songs, with your favorite first
- **Audio**: Take your best lyrics to [Suno](https://suno.com) and generate audio
- **Essay** (500–700 words): Explain your prompt chain, include sample prompts, and reflect on what GPT-4o got right and wrong about your poet